# ENEM 2024 · Sprint 2 — Engenharia de features

Entrada: `PARTICIPANTES_2024.csv` + `COMPETENCIAS_REDACAO_2024.csv` (merge por `NU_INSCRICAO`).
Saída: base de features com alvos, split treino/teste e as duas codificações (ordinal e one-hot).

A justificativa de cada feature está em `FEATURES.md`.

## 1 · Leitura e merge

In [1]:
import os, numpy as np, pandas as pd

pd.set_option("display.max_columns", 80); pd.set_option("display.width", 170)
SEED = 42

DATA_DIR = os.environ.get("ENEM_DIR", "/content")
OUT_DIR  = os.environ.get("ENEM_OUT", os.path.join(DATA_DIR, "data_processed"))
os.makedirs(OUT_DIR, exist_ok=True)

PERFIL   = ["TP_FAIXA_ETARIA","TP_SEXO","TP_ESTADO_CIVIL","TP_COR_RACA","TP_NACIONALIDADE",
            "TP_ST_CONCLUSAO","TP_ANO_CONCLUIU","TP_ENSINO","IN_TREINEIRO","SG_UF_PROVA"]
QUESTOES = [f"Q{i:03d}" for i in range(1, 24)]
CATEG    = PERFIL + QUESTOES

part = pd.read_csv(os.path.join(DATA_DIR, "PARTICIPANTES_2024.csv"), sep=";", encoding="latin1",
                   usecols=["NU_INSCRICAO","CO_MUNICIPIO_PROVA"] + CATEG,
                   dtype={c: "category" for c in CATEG})
red  = pd.read_csv(os.path.join(DATA_DIR, "COMPETENCIAS_REDACAO_2024.csv"), sep=";", encoding="latin1",
                   usecols=["NU_INSCRICAO","TP_STATUS_REDACAO","NU_NOTA_REDACAO"] +
                           [f"NU_NOTA_COMP{k}" for k in range(1, 6)],
                   dtype={c: "float32" for c in ["TP_STATUS_REDACAO","NU_NOTA_REDACAO"] +
                                                [f"NU_NOTA_COMP{k}" for k in range(1, 6)]})

df = part.merge(red, on="NU_INSCRICAO", how="outer", validate="one_to_one")
del part, red
print(df.shape)

(4332944, 42)


## 2 · Alvos

In [2]:
df["y_fez_redacao"] = df["NU_NOTA_REDACAO"].notna().astype("int8")
df["y_nota"]        = df["NU_NOTA_REDACAO"].astype("float32")                      # só quem fez
df["y_zerada"]      = np.where(df["y_fez_redacao"] == 1,
                               (df["TP_STATUS_REDACAO"] != 1).astype("float32"), np.nan).astype("float32")
df["y_nota_ok"]     = df["y_nota"].where(df["TP_STATUS_REDACAO"] == 1)             # só corrigidas sem problema
df["y_alto"]        = np.where(df["y_nota"].notna(), (df["y_nota"] >= 700).astype("float32"), np.nan).astype("float32")

ALVOS = ["y_fez_redacao", "y_nota", "y_zerada", "y_nota_ok", "y_alto"]
df.drop(columns=["NU_NOTA_REDACAO", "TP_STATUS_REDACAO"] + [f"NU_NOTA_COMP{k}" for k in range(1, 6)],
        inplace=True)
df[ALVOS].describe().round(3)

,y_fez_redacao,y_nota,y_zerada,y_nota_ok,y_alto
count,4332944.000,3167955.000,3167955.000,3002710.000,3167955.000
mean,0.731,624.593,0.052,658.965,0.393
std,0.443,216.343,0.222,163.491,0.488
min,0.000,0.000,0.000,40.000,0.000
25%,0.000,520.000,0.000,540.000,0.000
50%,1.000,640.000,0.000,640.000,0.000
75%,1.000,780.000,0.000,800.000,1.000
max,1.000,1000.000,1.000,1000.000,1.000


## 3 · Codificação ordinal do questionário

In [3]:
ESCOLARIDADE = {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5,"G":6}          # H = "não sei" -> NaN + flag
OCUPACAO     = {"A":1,"B":2,"C":3,"D":4,"E":5}                      # F = "não sei" -> NaN + flag
BIN          = {"A":0,"B":1}
QTD_4        = {"A":0,"B":1,"C":2,"D":3}
QTD_5        = {"A":0,"B":1,"C":2,"D":3,"E":4}
# ponto médio da faixa de renda em R$ (última faixa: 1,2x o limite inferior)
RENDA_MEIO   = {"A":0,"B":706,"C":1765,"D":2471,"E":3177,"F":3883,"G":4942,"H":6354,"I":7766,
                "J":9178,"K":10590,"L":12002,"M":13414,"N":15532,"O":19062,"P":24710,"Q":33888}
IDADE_MEIO   = {"1":16,"2":17,"3":18,"4":19,"5":20,"6":21,"7":22,"8":23,"9":24,"10":25,"11":28,
                "12":33,"13":38,"14":43,"15":48,"16":53,"17":58,"18":63,"19":68,"20":73}
ANOS_CONCL   = {"1":1,"2":2,"3":3,"4":4,"5":5,"6":6,"7":7,"8":8,"9":9,"10":10,"11":11,"12":12,
                "13":13,"14":14,"15":15,"16":16,"17":17,"18":19}                  # 0 = não informado -> NaN
REGIAO       = {**{uf:"Norte"    for uf in ["AC","AP","AM","PA","RO","RR","TO"]},
                **{uf:"Nordeste" for uf in ["AL","BA","CE","MA","PB","PE","PI","RN","SE"]},
                **{uf:"Centro-Oeste" for uf in ["DF","GO","MT","MS"]},
                **{uf:"Sudeste"  for uf in ["ES","MG","RJ","SP"]},
                **{uf:"Sul"      for uf in ["PR","RS","SC"]}}

def mapear(col, dicionario, tipo="float32"):
    return df[col].astype("object").map(dicionario).astype(tipo)

BENS_QTD  = {"Q008":QTD_4,"Q009":QTD_4,"Q010":QTD_4,"Q011":QTD_4,"Q012":QTD_4,"Q013":QTD_4,
             "Q018":QTD_4,"Q021":QTD_5,"Q022":QTD_5}
BENS_BIN  = {"Q014":BIN,"Q015":BIN,"Q016":BIN,"Q017":BIN,"Q019":BIN,"Q020":BIN}

for q, dic in {**BENS_QTD, **BENS_BIN}.items():
    df[f"n_{q.lower()}"] = mapear(q, dic)

df["esc_pai"]  = mapear("Q001", ESCOLARIDADE)
df["esc_mae"]  = mapear("Q002", ESCOLARIDADE)
df["ocup_pai"] = mapear("Q003", OCUPACAO)
df["ocup_mae"] = mapear("Q004", OCUPACAO)
df["pessoas_casa"]  = df["Q005"].astype("object").astype("float32")
df["tem_renda"]     = mapear("Q006", BIN)
df["renda_familiar"] = mapear("Q007", RENDA_MEIO)
df["idade"]          = mapear("TP_FAIXA_ETARIA", IDADE_MEIO)
df["anos_desde_conclusao"] = mapear("TP_ANO_CONCLUIU", ANOS_CONCL)

BENS = [f"n_{q.lower()}" for q in list(BENS_QTD) + list(BENS_BIN)]
df[["esc_pai","esc_mae","ocup_pai","ocup_mae","renda_familiar","pessoas_casa","idade"]].describe().round(2)

,esc_pai,esc_mae,ocup_pai,ocup_mae,renda_familiar,pessoas_casa,idade
count,3847266.00,4157560.00,3672515.00,3870663.00,4332944.00,4332944.00,4332944.00
mean,3.12,3.57,2.70,2.58,3420.15,3.70,21.03
std,1.63,1.56,1.25,1.21,5350.44,1.43,7.34
min,0.00,0.00,1.00,1.00,0.00,1.00,16.00
25%,2.00,3.00,2.00,2.00,706.00,3.00,17.00
50%,3.00,4.00,3.00,2.00,1765.00,4.00,18.00
75%,4.00,4.00,4.00,4.00,3883.00,4.00,21.00
max,6.00,6.00,5.00,5.00,33888.00,20.00,73.00


## 4 · Features derivadas

In [4]:
SM_2024 = 1412.0

# --- renda ---
df["renda_pc"]      = (df["renda_familiar"] / df["pessoas_casa"]).astype("float32")
df["log_renda_pc"]  = np.log1p(df["renda_pc"]).astype("float32")
df["renda_sm"]      = (df["renda_familiar"] / SM_2024).astype("float32")
df["sem_renda"]     = (df["renda_familiar"] == 0).astype("int8")

# --- escolaridade e ocupação dos pais ---
df["esc_max_pais"]   = df[["esc_pai","esc_mae"]].max(axis=1).astype("float32")
df["esc_media_pais"] = df[["esc_pai","esc_mae"]].mean(axis=1).astype("float32")
df["pais_superior"]  = (df["esc_max_pais"] >= 5).astype("int8")
df["nao_sabe_esc"]   = ((df["Q001"] == "H") | (df["Q002"] == "H")).astype("int8")
df["ocup_max_pais"]  = df[["ocup_pai","ocup_mae"]].max(axis=1).astype("float32")
df["nao_sabe_ocup"]  = ((df["Q003"] == "F") | (df["Q004"] == "F")).astype("int8")

# --- bens: índice por média de z-scores (acumulado coluna a coluna para poupar memória) ---
def media_z(colunas):
    acc = np.zeros(len(df), dtype="float32")
    for c in colunas:
        v = df[c].to_numpy(dtype="float32")
        acc += (v - np.nanmean(v)) / np.nanstd(v)
    return (acc / len(colunas)).astype("float32")

df["indice_bens"] = media_z(BENS)

# --- densidade domiciliar e acesso digital ---
df["comodos_por_pessoa"]     = ((df["n_q009"] + df["n_q010"]) / df["pessoas_casa"]).astype("float32")
df["computador_por_pessoa"]  = (df["n_q021"] / df["pessoas_casa"]).astype("float32")
df["celular_por_pessoa"]     = (df["n_q022"] / df["pessoas_casa"]).astype("float32")
df["tem_computador"]         = (df["n_q021"] > 0).astype("int8")
df["acesso_digital"]         = ((df["n_q021"] > 0) & (df["n_q020"] > 0)).astype("int8")

# --- índice socioeconômico composto (proxy do INSE) ---
df["inse_proxy"] = media_z(["log_renda_pc", "esc_max_pais", "indice_bens"])

# --- escola ---
df["escola_publica"]   = (df["Q023"] == "A").astype("int8")
df["escola_privada"]   = df["Q023"].isin(["D","E"]).astype("int8")
df["escola_mista"]     = df["Q023"].isin(["B","C"]).astype("int8")
df["bolsista"]         = df["Q023"].isin(["C","E"]).astype("int8")
df["sem_ensino_medio"] = (df["Q023"] == "F").astype("int8")

# --- trajetória escolar ---
df["treineiro"]   = df["IN_TREINEIRO"].astype("object").astype("int8")
df["concluinte"]  = (df["TP_ST_CONCLUSAO"] == "2").astype("int8")
df["ja_concluiu"] = (df["TP_ST_CONCLUSAO"] == "1").astype("int8")
df["atraso_idade"] = np.where(df["concluinte"] == 1, df["idade"] - 17, np.nan).astype("float32")
df["anos_desde_conclusao"] = df["anos_desde_conclusao"].fillna(0).astype("float32")

# --- demografia ---
df["sexo_fem"]   = (df["TP_SEXO"] == "F").astype("int8")
df["cor_raca"]   = df["TP_COR_RACA"].astype("object").map(
    {"0":"nao_declarado","1":"branca","2":"preta","3":"parda","4":"amarela","5":"indigena","6":"sem_info"}
).astype("category")
df["regiao"]     = df["SG_UF_PROVA"].astype("object").map(REGIAO).astype("category")
df["uf"]         = df["SG_UF_PROVA"].astype("category")

print(df.shape)

(4332944, 94)


## 5 · Split treino/teste (estratificado por UF)

In [5]:
rng = np.random.default_rng(SEED)
u = rng.random(len(df)).astype("float32")
corte = pd.Series(u).groupby(df["uf"].to_numpy(), observed=True).quantile(0.8)
df["split"] = np.where(u <= df["uf"].astype("object").map(corte).to_numpy(dtype="float32"),
                       "treino", "teste")
del u
treino = df["split"] == "treino"

pd.crosstab(df["uf"], df["split"], normalize="index").round(3).head()

split,teste,treino
uf,,
AC,0.2,0.8
AL,0.2,0.8
AM,0.2,0.8
AP,0.2,0.8
BA,0.2,0.8


## 6 · Encodings de município e UF (ajustados só no treino)

In [6]:
# frequência: porte do município como local de aplicação
freq = df.loc[treino, "CO_MUNICIPIO_PROVA"].value_counts()
df["log_porte_municipio"] = np.log1p(df["CO_MUNICIPIO_PROVA"].map(freq).fillna(0)).astype("float32")

# target encoding suavizado da nota, calculado apenas nas linhas de treino
def target_encode(chave, alvo="y_nota", m=200):
    aux = df.loc[treino & df[alvo].notna(), [chave, alvo]]
    media_geral = aux[alvo].mean()
    g = aux.groupby(chave, observed=True)[alvo].agg(["mean", "count"])
    suave = (g["mean"] * g["count"] + media_geral * m) / (g["count"] + m)
    return df[chave].astype("object").map(suave).fillna(media_geral).astype("float32"), media_geral

df["te_municipio"], _ = target_encode("CO_MUNICIPIO_PROVA")
df["te_uf"], _        = target_encode("uf")

df.loc[treino, ["log_porte_municipio","te_municipio","te_uf"]].describe().round(2)

,log_porte_municipio,te_municipio,te_uf
count,3466351.00,3466351.00,3466351.00
mean,8.59,625.57,623.76
std,1.62,37.76,28.28
min,3.69,477.27,540.72
25%,7.28,599.24,598.39
50%,8.30,627.03,624.09
75%,9.75,652.69,650.74
max,11.81,713.07,664.96


## 7 · Matriz de features

In [7]:
NUM = ["renda_familiar","renda_pc","log_renda_pc","renda_sm","sem_renda","tem_renda",
       "pessoas_casa","esc_pai","esc_mae","esc_max_pais","esc_media_pais","pais_superior","nao_sabe_esc",
       "ocup_pai","ocup_mae","ocup_max_pais","nao_sabe_ocup",
       "indice_bens","inse_proxy","comodos_por_pessoa","computador_por_pessoa","celular_por_pessoa",
       "tem_computador","acesso_digital",
       "escola_publica","escola_privada","escola_mista","bolsista","sem_ensino_medio",
       "idade","atraso_idade","anos_desde_conclusao","treineiro","concluinte","ja_concluiu",
       "sexo_fem","log_porte_municipio","te_municipio","te_uf"] + BENS
CAT = ["cor_raca","regiao","uf"]
FEATURES = NUM + CAT

import gc
df.drop(columns=[c for c in CATEG + ["CO_MUNICIPIO_PROVA"] if c in df.columns], inplace=True)
gc.collect()
base = df
print("linhas:", len(base), "| features:", len(FEATURES))
base[NUM].isna().mean().round(4).sort_values(ascending=False).head(8)

linhas: 4332944 | features: 57


atraso_idade      0.6267
ocup_pai          0.1524
esc_pai           0.1121
ocup_mae          0.1067
ocup_max_pais     0.0504
esc_mae           0.0405
esc_media_pais    0.0292
esc_max_pais      0.0292
dtype: float64

In [8]:
# imputação: só onde o NaN não tem significado próprio (medianas do treino)
med = base.loc[base["split"] == "treino", NUM].median()
AUSENTES = ["esc_pai","esc_mae","esc_max_pais","esc_media_pais","ocup_pai","ocup_mae",
            "ocup_max_pais","atraso_idade"]
for c in AUSENTES:
    base[f"{c}_ausente"] = base[c].isna().astype("int8")
for c in NUM:
    if base[c].isna().any():
        base[c] = base[c].fillna(med[c])
    base[c] = base[c].astype("float32")
NUM = NUM + [f"{c}_ausente" for c in AUSENTES]
FEATURES = NUM + CAT
gc.collect()
print("NaN restantes nas features:", int(base[FEATURES].isna().sum().sum()))

NaN restantes nas features: 0


## 8 · Verificação

In [9]:
amostra_corr = base.loc[base["y_nota"].notna()].sample(300_000, random_state=SEED)
corr = amostra_corr[NUM].corrwith(amostra_corr["y_nota"]).sort_values(key=abs, ascending=False).round(3)
del amostra_corr
display(corr.head(15).to_frame("corr_com_nota"))

print("treino/teste:", base["split"].value_counts(normalize=True).round(3).to_dict())
print("alvos disponíveis:", {c: int(base[c].notna().sum()) for c in ALVOS})
assert base["NU_INSCRICAO"].is_unique
assert int(base[FEATURES].isna().sum().sum()) == 0
assert all(np.isfinite(base[c].to_numpy()).all() for c in NUM)
print("verificações OK")

,corr_com_nota
inse_proxy,0.321
escola_privada,0.315
escola_publica,-0.299
esc_media_pais,0.280
esc_max_pais,0.272
n_q021,0.272
ocup_max_pais,0.271
log_renda_pc,0.264
pais_superior,0.260
ocup_mae,0.258


treino/teste: {'treino': 0.8, 'teste': 0.2}
alvos disponíveis: {'y_fez_redacao': 4332944, 'y_nota': 3167955, 'y_zerada': 3167955, 'y_nota_ok': 3002710, 'y_alto': 3167955}


verificações OK


## 9 · Versão one-hot (amostra)

In [10]:
amostra = base.sample(500_000, random_state=SEED)
X_ohe = pd.get_dummies(amostra[FEATURES], columns=CAT, drop_first=True, dtype="float32")
ohe = pd.concat([amostra[["NU_INSCRICAO","split"] + ALVOS], X_ohe], axis=1)
print("amostra one-hot:", ohe.shape)

amostra one-hot: (500000, 104)


## 10 · Gravação

In [11]:
def salvar(tabela, nome, chunk=1_000_000):
    # grava em parquet por blocos: evita pico de memória com a base cheia
    caminho = os.path.join(OUT_DIR, nome + ".parquet")
    try:
        import pyarrow as pa, pyarrow.parquet as pq
        writer = None
        for ini in range(0, len(tabela), chunk):
            t = pa.Table.from_pandas(tabela.iloc[ini:ini + chunk], preserve_index=False)
            writer = writer or pq.ParquetWriter(caminho, t.schema, compression="snappy")
            writer.write_table(t)
        writer.close()
    except Exception as e:
        caminho = os.path.join(OUT_DIR, nome + ".csv.gz")
        tabela.to_csv(caminho, index=False, compression="gzip")
        print("parquet indisponível (%s); gravado como csv.gz" % type(e).__name__)
    print(f"{nome}: {tabela.shape} -> {caminho} ({os.path.getsize(caminho)/1e6:.1f} MB)")

salvar(base, "enem2024_features_ordinal")
salvar(ohe,  "enem2024_features_amostra_onehot")
pd.Series(FEATURES).to_csv(os.path.join(OUT_DIR, "lista_features.csv"), index=False, header=["feature"])

enem2024_features_ordinal: (4332944, 72) -> /sessions/rcw-01cky24mjxfql3xtvv4q9vpb/mnt/INF1032 - DataScience/a/inf1032_3wa05/data_processed/enem2024_features_ordinal.parquet (149.3 MB)


enem2024_features_amostra_onehot: (500000, 104) -> /sessions/rcw-01cky24mjxfql3xtvv4q9vpb/mnt/INF1032 - DataScience/a/inf1032_3wa05/data_processed/enem2024_features_amostra_onehot.parquet (18.8 MB)


## Como carregar no próximo sprint

```python
base = pd.read_parquet("data_processed/enem2024_features_ordinal.parquet")
treino, teste = base[base.split == "treino"], base[base.split == "teste"]
X_tr, y_tr = treino[FEATURES], treino["y_nota"]
```

`te_municipio` e `te_uf` foram calculados só com linhas de treino; se o split mudar, refaça o passo 6.